# Seção 4 — Inferência local, remota ou privada

## 1. Configuração inicial

In [1]:
from pathlib import Path
import os
import sys
import json
import time
import platform
from datetime import datetime

import pandas as pd
from gpt4all import GPT4All

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

MODEL_DIR = ROOT_DIR / "models"
OUTPUT_DIR = ROOT_DIR / "outputs" / "avaliacoes"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Sistema:", platform.platform())
print("Raiz do projeto:", ROOT_DIR)

Python: C:\Users\breno\miniconda3\envs\historia-rag\python.exe
Sistema: Windows-10-10.0.26200-SP0
Raiz do projeto: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment


## 2. Estratégia escolhida

In [2]:
estrategia_inferencia = {
    "estrategia_escolhida": "inferência local com GPT4All",
    "modelo": "Phi-3-mini-4k-instruct.Q4_0.gguf",
    "motivo": (
        "A execução local foi escolhida para reduzir dependência de APIs externas, "
        "evitar exposição do corpus a provedores cloud e manter controle sobre a execução."
    ),
    "api_externa_utilizada": False,
    "usa_chaves_ou_tokens": False
}

estrategia_inferencia

{'estrategia_escolhida': 'inferência local com GPT4All',
 'modelo': 'Phi-3-mini-4k-instruct.Q4_0.gguf',
 'motivo': 'A execução local foi escolhida para reduzir dependência de APIs externas, evitar exposição do corpus a provedores cloud e manter controle sobre a execução.',
 'api_externa_utilizada': False,
 'usa_chaves_ou_tokens': False}

## 3. Verificação de GPU e fallback para CPU

In [3]:
MODEL_NAME = "Phi-3-mini-4k-instruct.Q4_0.gguf"

try:
    gpus_disponiveis = GPT4All.list_gpus()
except Exception as erro:
    gpus_disponiveis = []
    print("Não foi possível listar GPUs:", erro)

print("GPUs disponíveis:", gpus_disponiveis)

device_escolhido = "gpu" if len(gpus_disponiveis) > 0 else "cpu"
print("Dispositivo inicialmente escolhido:", device_escolhido)

GPUs disponíveis: ['kompute:NVIDIA GeForce RTX 4070']
Dispositivo inicialmente escolhido: gpu


## 4. Carregamento do modelo local

In [4]:
inicio_carregamento = time.perf_counter()

try:
    model = GPT4All(
        model_name=MODEL_NAME,
        model_path=str(MODEL_DIR),
        allow_download=True,
        n_threads=os.cpu_count(),
        n_ctx=2048,
        device=device_escolhido
    )

except Exception as erro:
    print("Falha ao carregar em GPU. Tentando carregar em CPU.")
    print("Erro:", erro)

    device_escolhido = "cpu"

    model = GPT4All(
        model_name=MODEL_NAME,
        model_path=str(MODEL_DIR),
        allow_download=True,
        n_threads=os.cpu_count(),
        n_ctx=2048,
        device=device_escolhido
    )

tempo_carregamento = round(time.perf_counter() - inicio_carregamento, 2)

print("Modelo carregado:", MODEL_NAME)
print("Executando em:", device_escolhido)
print("Tempo de carregamento:", tempo_carregamento, "segundos")

Modelo carregado: Phi-3-mini-4k-instruct.Q4_0.gguf
Executando em: gpu
Tempo de carregamento: 1.59 segundos


## 5. Teste de inferência local

In [5]:
SYSTEM_PROMPT = """
Você é um assistente técnico e histórico do projeto historIA-RAG.
Neste notebook, "inferência local" significa executar o modelo de linguagem no computador do usuário, sem API externa.
RAG significa Retrieval-Augmented Generation: recuperação de trechos do corpus antes da geração da resposta.
Responda em português brasileiro, de forma objetiva e acadêmica.
Não use outros significados para "inferência local" ou "RAG".
"""

def gerar_resposta(prompt: str, max_tokens: int = 250, temp: float = 0.2) -> str:
    prompt_completo = f"""
{SYSTEM_PROMPT}

Tarefa do usuário:
{prompt}

Resposta:
"""
    with model.chat_session():
        resposta = model.generate(
            prompt_completo,
            max_tokens=max_tokens,
            temp=temp,
            top_k=40,
            top_p=0.9,
            repeat_penalty=1.15
        )
    return resposta.strip()

pergunta_teste = (
    "Explique em um parágrafo por que executar o GPT4All localmente pode ser útil "
    "em um sistema RAG com corpus próprio, considerando privacidade, custo e dependência de internet."
)

inicio_inferencia = time.perf_counter()
resposta_teste = gerar_resposta(pergunta_teste, max_tokens=220, temp=0.2)
tempo_inferencia = round(time.perf_counter() - inicio_inferencia, 2)

print("Pergunta:", pergunta_teste)
print("\nResposta:\n", resposta_teste)
print("\nTempo de inferência:", tempo_inferencia, "segundos")


Pergunta: Explique em um parágrafo por que executar o GPT4All localmente pode ser útil em um sistema RAG com corpus próprio, considerando privacidade, custo e dependência de internet.

Resposta:
 Executar o modelo GPT-4All localmente no contexto de um Sistema de Generação Aumentada por Recuperação (RAG) com seu próprio corpus pode trazer benefícios significativos, especialmente em relação à privacidade dos dados. O processamento do modelo a borda permite que os trechos relevantes sejam recuperados e utilizados de forma segura dentro da rede local, evitando assim o risco associado ao envio desses dados para servidores externos. Além disso, uma inferência local reduz consideravelmente custos operacionais decorrentes do uso de internet, pois diminui a necessidade de conexão constante e as despesas relacionadas às taxas de transferência de grandes volumes de dados. Por fim, o RAG com capacidades locais também oferece maior confiabilidade na geração de respostas, independentemente da dispon

## 6. Dimensões avaliadas

In [6]:
avaliacao_dimensoes = [
    {
        "dimensao": "privacidade",
        "avaliacao": "favorável",
        "justificativa": "As perguntas e trechos do corpus são processados localmente, sem envio obrigatório a APIs externas."
    },
    {
        "dimensao": "custo",
        "avaliacao": "favorável",
        "justificativa": "Após o download do modelo, não há custo por chamada de API."
    },
    {
        "dimensao": "latência",
        "avaliacao": "variável",
        "justificativa": "A latência depende do hardware local. GPU tende a ser mais rápida; CPU é mais reprodutível, porém mais lenta."
    },
    {
        "dimensao": "disponibilidade",
        "avaliacao": "favorável",
        "justificativa": "Após baixar o modelo, a execução pode ocorrer localmente, com menor dependência de internet."
    },
    {
        "dimensao": "controle",
        "avaliacao": "favorável",
        "justificativa": "O projeto controla modelo, parâmetros, diretórios, prompts e formato de resposta."
    },
    {
        "dimensao": "facilidade de integração",
        "avaliacao": "adequada",
        "justificativa": "O pacote gpt4all permite integração direta em Python e Jupyter Lab."
    },
    {
        "dimensao": "limitações de hardware",
        "avaliacao": "atenção",
        "justificativa": "Modelos locais dependem de memória, CPU/GPU e compatibilidade do dispositivo."
    },
    {
        "dimensao": "dependência de internet",
        "avaliacao": "baixa após download",
        "justificativa": "A internet é necessária para baixar o modelo inicialmente, mas não para cada inferência."
    },
    {
        "dimensao": "risco de exposição de dados",
        "avaliacao": "reduzido",
        "justificativa": "Como não há API externa nesta etapa, o risco de envio acidental de corpus, prompts ou chaves é menor."
    }
]

df_dimensoes = pd.DataFrame(avaliacao_dimensoes)
pd.set_option("display.max_colwidth", None)
display(df_dimensoes)

,dimensao,avaliacao,justificativa
0,privacidade,favorável,"As perguntas e trechos do corpus são processados localmente, sem envio obrigatório a APIs externas."
1,custo,favorável,"Após o download do modelo, não há custo por chamada de API."
2,latência,variável,"A latência depende do hardware local. GPU tende a ser mais rápida; CPU é mais reprodutível, porém mais lenta."
3,disponibilidade,favorável,"Após baixar o modelo, a execução pode ocorrer localmente, com menor dependência de internet."
4,controle,favorável,"O projeto controla modelo, parâmetros, diretórios, prompts e formato de resposta."
5,facilidade de integração,adequada,O pacote gpt4all permite integração direta em Python e Jupyter Lab.
6,limitações de hardware,atenção,"Modelos locais dependem de memória, CPU/GPU e compatibilidade do dispositivo."
7,dependência de internet,baixa após download,"A internet é necessária para baixar o modelo inicialmente, mas não para cada inferência."
8,risco de exposição de dados,reduzido,"Como não há API externa nesta etapa, o risco de envio acidental de corpus, prompts ou chaves é menor."


## 7. Comparação local versus API cloud

In [7]:
comparacao_execucao = [
    {
        "criterio": "Privacidade",
        "GPT4All local": "Maior controle local dos dados",
        "API cloud": "Dados enviados ao provedor externo"
    },
    {
        "criterio": "Custo",
        "GPT4All local": "Sem custo por chamada após download",
        "API cloud": "Pode haver custo por uso/tokens"
    },
    {
        "criterio": "Latência",
        "GPT4All local": "Depende do hardware local",
        "API cloud": "Depende da rede e da infraestrutura do provedor"
    },
    {
        "criterio": "Qualidade do modelo",
        "GPT4All local": "Limitada pelo modelo local escolhido",
        "API cloud": "Pode oferecer modelos maiores e mais robustos"
    },
    {
        "criterio": "Disponibilidade",
        "GPT4All local": "Pode funcionar offline após download",
        "API cloud": "Depende de internet e disponibilidade do serviço"
    },
    {
        "criterio": "Segurança de chaves",
        "GPT4All local": "Não exige chave de API",
        "API cloud": "Exige cuidado com tokens e variáveis de ambiente"
    }
]

df_comparacao = pd.DataFrame(comparacao_execucao)
display(df_comparacao)

,criterio,GPT4All local,API cloud
0,Privacidade,Maior controle local dos dados,Dados enviados ao provedor externo
1,Custo,Sem custo por chamada após download,Pode haver custo por uso/tokens
2,Latência,Depende do hardware local,Depende da rede e da infraestrutura do provedor
3,Qualidade do modelo,Limitada pelo modelo local escolhido,Pode oferecer modelos maiores e mais robustos
4,Disponibilidade,Pode funcionar offline após download,Depende de internet e disponibilidade do serviço
5,Segurança de chaves,Não exige chave de API,Exige cuidado com tokens e variáveis de ambiente


## 8. Verificação de cuidados de segurança

In [8]:
cuidados_seguranca = {
    "usa_api_externa": False,
    "necessita_chave_api": False,
    "modelo_local_em_git": False,
    "diretorio_modelos": str(MODEL_DIR),
    "observacao": (
        "O diretório models/ e arquivos .gguf devem permanecer fora do GitHub. "
        "O projeto deve manter .env, chaves, tokens e credenciais no .gitignore."
    )
}

cuidados_seguranca

{'usa_api_externa': False,
 'necessita_chave_api': False,
 'modelo_local_em_git': False,
 'diretorio_modelos': 'C:\\Users\\breno\\Desktop\\Sistemas Cognitivos com LLMs - Assessment\\models',
 'observacao': 'O diretório models/ e arquivos .gguf devem permanecer fora do GitHub. O projeto deve manter .env, chaves, tokens e credenciais no .gitignore.'}

## 9. Salvamento dos resultados

In [9]:
resultados_c04 = {
    "data_execucao": datetime.now().isoformat(),
    "estrategia_inferencia": estrategia_inferencia,
    "ambiente": {
        "python": sys.executable,
        "sistema": platform.platform(),
        "cpu_threads": os.cpu_count(),
        "gpus_disponiveis": gpus_disponiveis,
        "device_escolhido": device_escolhido
    },
    "modelo": {
        "nome": MODEL_NAME,
        "diretorio": str(MODEL_DIR),
        "tempo_carregamento_segundos": tempo_carregamento
    },
    "teste_inferencia": {
        "pergunta": pergunta_teste,
        "resposta": resposta_teste,
        "tempo_inferencia_segundos": tempo_inferencia
    },
    "dimensoes_avaliadas": avaliacao_dimensoes,
    "comparacao_local_cloud": comparacao_execucao,
    "cuidados_seguranca": cuidados_seguranca
}

json_saida = OUTPUT_DIR / "c04_inferencia_resultados.json"
csv_dimensoes = OUTPUT_DIR / "c04_inferencia_dimensoes.csv"
csv_comparacao = OUTPUT_DIR / "c04_inferencia_comparacao.csv"

with json_saida.open("w", encoding="utf-8") as f:
    json.dump(resultados_c04, f, ensure_ascii=False, indent=2)

df_dimensoes.to_csv(csv_dimensoes, index=False, encoding="utf-8-sig")
df_comparacao.to_csv(csv_comparacao, index=False, encoding="utf-8-sig")

print("JSON salvo em:", json_saida)
print("CSV dimensões salvo em:", csv_dimensoes)
print("CSV comparação salvo em:", csv_comparacao)

JSON salvo em: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c04_inferencia_resultados.json
CSV dimensões salvo em: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c04_inferencia_dimensoes.csv
CSV comparação salvo em: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c04_inferencia_comparacao.csv


## 10. Fechamento do modelo

In [10]:
model.close()
print("Modelo fechado.")

Modelo fechado.
